
#### Bronze XML Ingestion script:




In [0]:
import os
import json
from pyspark.sql import functions as F

# --- 1. CONFIGURATION & WIDGETS ---
dbutils.widgets.text("datasets_json", "[]", "Datasets List (JSON Array)")

try:
    raw_input = dbutils.widgets.get("datasets_json")
    dataset_list = json.loads(raw_input)
    
    SOURCE_BASE = "/Volumes/data_landing/data_raw"
    DEST_CATALOG = "data_bronze"
    DEST_SCHEMA = "bronze"
    
except Exception as e:
    print(f"Setup Error: {str(e)}")
    raise

# --- 2. MODULAR XML INGESTION COMPONENT ---

def ingest_xml_to_bronze(dataset_name):
    """
    Ingests chunk4.xml and appends it to the existing Bronze Delta table.
    """
    try:
        # Define the file path
        source_file = f"{SOURCE_BASE}/{dataset_name}/chunks/chunk4.xml"
        target_table = f"{DEST_CATALOG}.{DEST_SCHEMA}.{dataset_name.lower()}"
        
        # --- FIX: Define XML_PATH ---
        XML_PATH = source_file
        
        if not os.path.exists(source_file):
            print(f"Skipping: {dataset_name} (No XML chunk found at {source_file})")
            return

        print(f"Ingesting XML: {dataset_name} -> {target_table}")

        # 2. READ XML WITH EXPLICIT NESTING
        # Note: If your structure is <root><records><item>, rowTag must be "item"
        df_xml = (spark.read
                  .format("xml")
                  .option("rowTag", "item") 
                  .load(XML_PATH))

        # Check if Spark picked up the columns correctly
        if len(df_xml.columns) == 0 or df_xml.count() == 0:
            print(f"!!! WARNING: No records found in {source_file} using 'item' tag.")
            # FALLBACK: Try the structure you had previously or 'records', minor confusion between tags for records
            # or item
            print("Retrying with broader inference...")
            df_xml = spark.read.format("xml").option("rowTag", "Record").load(XML_PATH)

        # 3. CAST TO STRING & ADD AUDIT COLUMNS
        # Ensuring 100% String compliance for Bronze layer
        final_df = (df_xml.select([F.col(c).cast("string") for c in df_xml.columns])
                          .withColumn("load_dt", F.current_timestamp())
                          .withColumn("source", F.lit("chunk4.xml")))

        row_count = final_df.count()
        print(f"Verified rows for {dataset_name}: {row_count}")

        # 4. APPEND TO DELTA TABLE
        (final_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(target_table))
        
        print(f"SUCCESS: Appended XML data to {target_table}")

    except Exception as e:
        print(f"ERROR: Failed processing {dataset_name}: {str(e)}")

# --- 3. EXECUTION ---
if __name__ == "__main__":
    if not dataset_list:
        print("No datasets provided.")
    else:
        for ds in dataset_list:
            ingest_xml_to_bronze(ds)

### Unit testing:

As per Deliverable Standards, we must verify that the XML records were correctly integrated with the CSV records.

In [0]:
def validate_bronze_layer(target_datasets):
    """
    Standardizes validation across all ingested Bronze tables.
    Validates record counts, string-only typing, and audit columns.
    """
    print(f"Validation started for {len(target_datasets)} datasets.")
    print("-" * 60)
    
    for ds in target_datasets:
        table_full_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{ds.lower()}"
        
        try:
            # 1. Verify Table Existence
            assert spark.catalog.tableExists(table_full_name), f"Table {table_full_name} not found."
            
            # 2. Filter for the specific XML ingestion source
            test_df = spark.table(table_full_name).filter(F.col("source") == "chunk4.xml")
            row_count = test_df.count()
            
            # 3. Validation: Data Presence
            assert row_count > 0, f"Zero XML records found in {table_full_name}."
            
            # 4. Validation: Column Data Types (Bronze Requirement)
            # Ensure every column is String except the system generated load_dt
            for field in test_df.schema:
                if field.name != "load_dt":
                    assert str(field.dataType) == "StringType()", \
                        f"Type mismatch in {ds}: {field.name} is {field.dataType}, expected StringType."
            
            # 5. Validation: Audit Columns
            current_cols = test_df.columns
            assert "load_dt" in current_cols and "source" in current_cols, \
                f"Missing audit columns in {ds}."
            
            print(f"PASSED: {ds.ljust(25)} | Rows: {row_count}")
            
        except AssertionError as ae:
            print(f"FAILED: {ds.ljust(25)} | Reason: {str(ae)}")
            # Optional: raise Exception(ae) to stop the Databricks Job here
        except Exception as e:
            print(f"ERROR:  {ds.ljust(25)} | Technical Error: {str(e)[:50]}")

    print("-" * 60)
    print("Bronze validation process completed.")

# --- EXECUTION ---
if __name__ == "__main__":
    if dataset_list:
        validate_bronze_layer(dataset_list)
    else:
        print("No datasets provided for validation.")

# Validation failed for datasets which were not chunked and 2 are chunked so they pass the condition